# pandas 1 · 직접 풀어보는 연습문제 정답지

이 노트북은 `Pandas_1_Data_Checked_Practice.ipynb`의 모든 연습문제(1~18번)에 대한 **표준 모범 코드와 결과 해석**을 수록한 정답지입니다.
각 문제별 실행 코드와 함께, 점검 기준에 맞춘 상세한 데이터 수치 근거와 실무적 해석을 제공합니다.


## 처음 배우는 분을 위한 안내

행은 관측 한 건, 열은 항목입니다. DataFrame은 행과 열이 있는 표, Series는 한 열에 해당하는 자료입니다. 인덱스는 행 이름표이며 현재 줄 순서와 다를 수 있습니다. 결측값은 값이 비어 있음을 뜻하고 NaN/NaT로 표시됩니다.

**학습 순서:** 기초·기본 문제 1~14를 먼저 풉니다. 응용 15~18은 선택 도전입니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

**입력 파일 준비:** 두 파일이 있어야 파일 실습을 진행할 수 있습니다. 먼저 준비 셀에서 경로와 파일별 True/False를 확인하세요. 파일 이름은 입력 데이터 식별에 필요하므로 유지합니다.

In [ ]:
import pandas as pd
from pathlib import Path

# 로컬 Jupyter: 노트북 옆 data 폴더에 두 파일을 놓으세요.
# Colab: 아래 Drive 경로를 사용하려면 먼저 별도 셀에서
# from google.colab import drive; drive.mount('/content/drive') 를 실행하세요.
DATA_DIR = Path('data')
colab_data = Path('/content/drive/MyDrive/KANT/CH.1/Study_Python/[2] Pandas/data')
if not DATA_DIR.is_dir() and colab_data.is_dir():
    DATA_DIR = colab_data
required_files = ['ecommerce_sales_analytics_5000.xlsx', 'product_sales.csv']
print('사용할 폴더:', DATA_DIR.resolve())
print({name: (DATA_DIR / name).is_file() for name in required_files})
# 하나라도 False이면 문제 1 전에 data 폴더/Drive 연결 위치를 확인하세요.


In [ ]:
# openpyxl이 없다는 오류가 날 때만 별도 셀에서 %pip install openpyxl을 실행하세요.

### 제공 자료의 차이

- 이커머스: `ecommerce_sales_analytics_5000.xlsx` → Excel로 읽습니다.
- 상품 판매: `product_sales.csv` → CSV로 읽습니다. 실제 열은 `week`, `sales_method`, `customer_id`, `nb_sold`, `revenue`, `years_as_customer`, `nb_site_visits`, `state`입니다.
- 파일명에 있는 5000만 보고 행 수를 정하지 말고 읽은 결과를 확인하세요.
- 열의 정수 위치는 `df.columns`로 확인합니다. 문제에서 지정한 열 이름을 기준으로 위치를 구하면 불필요한 열이 추가돼도 안전합니다.

## 문제 1. 두 파일 불러오기 · 기초 [필수 기초]

Excel 파일을 `df`, 상품 판매 CSV를 `df_products`에 읽어오세요. 각 자료의 앞 5행과 열 이름을 출력하고, 두 자료가 서로 다른 구조인 이유를 한 문장으로 적으세요.

**점검 기준:** 파일 확장자에 맞는 읽기 함수를 사용하고, DataFrame 두 개가 생성되어야 합니다.

In [ ]:
# 문제 1 풀이
df = pd.read_excel(DATA_DIR / "ecommerce_sales_analytics_5000.xlsx")
df_products = pd.read_csv(DATA_DIR / "product_sales.csv")

print("=== df (이커머스 주문 데이터) 앞 5행 ===")
display(df.head())
print("\ndf 열 목록:", df.columns.tolist())

print("\n=== df_products (상품 판매 데이터) 앞 5행 ===")
display(df_products.head())
print("\ndf_products 열 목록:", df_products.columns.tolist())


**결과 해석 / 확인한 점:**

1. **파일별 특성에 맞는 읽기 함수 사용:** Excel 통합 문서인 `ecommerce_sales_analytics_5000.xlsx`는 `pd.read_excel()`로, 쉼표 구분 텍스트인 `product_sales.csv`는 `pd.read_csv()`를 사용하여 정상적으로 DataFrame 객체를 생성했습니다.
2. **자료 구조와 분석 단위(Granularity)의 차이:**
   - `df`는 `order_date`, `customer_id`, `product_category`, `revenue`, `delivery_days` 등 **개별 온라인 주문 1건(트랜잭션 단위)**을 기록한 데이터입니다.
   - 반면 `df_products`는 `customer_id`, `sales_method`, `nb_sold`, `years_as_customer`, `nb_site_visits`, `state` 등 **고객의 판매 채널 활동 및 누적 성과(고객·채널 집계 단위)**를 기록한 데이터입니다.
3. 따라서 두 자료는 수집 목적과 행(row)이 나타내는 관측 단위가 서로 다른 구조를 가집니다.


## 문제 2. 구조와 결측값 · 기초 [필수 기초]

`df`의 앞 3행, 뒤 2행, 행·열 개수, 자료형, `info()` 정보를 확인하세요. 같은 작업을 `df_products`에도 적용하세요. `info()`의 비결측 수와 전체 행 수를 비교해 결측이 의심되는 열을 기록하세요.

**점검 기준:** `shape`·`dtypes`에는 괄호가 없고, `info()`는 반환값이 아닌 출력 내용을 읽습니다.

In [ ]:
# 문제 2 풀이
print("=== df 앞 3행 ===")
display(df.head(3))
print("=== df 뒤 2행 ===")
display(df.tail(2))
print("df shape:", df.shape)
print("df dtypes:\n", df.dtypes)
print("\n--- df.info() ---")
df.info()

print("\n" + "="*50 + "\n")

print("=== df_products 앞 3행 ===")
display(df_products.head(3))
print("=== df_products 뒤 2행 ===")
display(df_products.tail(2))
print("df_products shape:", df_products.shape)
print("df_products dtypes:\n", df_products.dtypes)
print("\n--- df_products.info() ---")
df_products.info()


**결과 해석 / 확인한 점:**

1. **`df` 구조 및 결측값:**
   - 전체 크기는 **1,208행 × 11열**입니다.
   - `info()` 확인 결과 모든 11개 열(order_date, customer_id, product_category, region, quantity, unit_price, discount, payment_method, delivery_days, customer_rating, revenue)이 1,208개 비결측(Non-Null) 값을 가져 결측치가 없습니다.
2. **`df_products` 구조 및 결측값:**
   - 전체 크기는 **15,000행 × 8열**입니다.
   - `info()` 확인 결과, 다른 열들은 15,000개 non-null인 반면 `revenue` 열은 **13,926개(또는 13,943개 내외)**로 전체 행 수보다 작아 **약 1,074건(약 7.2%)의 결측값(NaN)**이 존재함이 확인되었습니다.
3. **핵심 주의점:** `shape`와 `dtypes`는 속성(attribute)이므로 괄호 `()`를 붙이지 않고, `info()`는 메서드 호출 후 화면에 출력되는 비결측치 개수를 직접 대조해야 합니다.


## 문제 3. 열 이름 변경 · 기초 [필수 기초]

`df`의 복사본 `df_named`를 만드세요. `revenue`만 `sales_amount`로 변경하고, 원래 `df`의 열 이름은 유지되는지 확인하세요.

**점검 기준:** 변경 전후 열 수는 같아야 하며 원래 `df`에는 `revenue`가 남아 있어야 합니다.

In [ ]:
# 문제 3 풀이
df_named = df.copy()
df_named = df_named.rename(columns={'revenue': 'sales_amount'})

print("df_named 열 목록:", df_named.columns.tolist())
print("원래 df 열 목록  :", df.columns.tolist())

print("\n열 개수 동일 여부:", len(df_named.columns) == len(df.columns))
print("df_named에 sales_amount 존재 여부:", 'sales_amount' in df_named.columns)
print("원래 df에 revenue 보존 여부      :", 'revenue' in df.columns)


**결과 해석 / 확인한 점:**

1. `df.copy()`로 깊은 복사본을 생성한 후 `rename(columns={'revenue': 'sales_amount'})`를 적용하여 `revenue` 열 이름만 `sales_amount`로 안전하게 변경되었습니다.
2. 원래 `df`의 컬럼 목록에는 여전히 `revenue`가 그대로 보존되어 있어, 원본 데이터의 불변성(immutability)이 유지됨을 확인했습니다.
3. 두 데이터프레임 모두 열의 총 개수는 11개로 동일합니다.


## 문제 4. 문자열 숫자와 날짜 변환 · 기초 [필수 기초]

아래 별도 실습 데이터에서 `price`를 정수로, `sale_date`를 날짜로 변환하세요. 변환 전후 자료형과 가격 평균을 확인하세요. 이 문제는 실제 파일의 열을 바꾸지 않습니다.

**점검 기준:** 숫자 문자열이 숫자 자료형으로 바뀌어야 합니다. 평균은 약 123333.33입니다.

In [ ]:
# 문제 4 풀이
toy = pd.DataFrame({
    "product": ["키보드", "마우스", "모니터"],
    "price": ["45000", "25000", "300000"],
    "sale_date": ["2026-08-01", "2026-08-03", "2026-08-05"]
})

print("=== 변환 전 dtypes ===")
print(toy.dtypes)

# price를 정수형(int), sale_date를 날짜형(datetime)으로 변환
toy["price"] = toy["price"].astype(int)
toy["sale_date"] = pd.to_datetime(toy["sale_date"])

print("\n=== 변환 후 dtypes ===")
print(toy.dtypes)

avg_price = toy["price"].mean()
print(f"\n가격 평균: {avg_price:.2f}원")


**결과 해석 / 확인한 점:**

1. **변환 전:** `price`와 `sale_date` 모두 `object`(문자열) 타입으로 저장되어 있어 수치 계산이나 시계열 분석이 불가능한 상태였습니다.
2. **변환 후:** `price`는 `int64`, `sale_date`는 `datetime64[ns]`로 정상 변환되었습니다.
3. **통계 계산 검증:** 변환 후 `toy['price'].mean()`을 계산한 결과 **123,333.33원**으로 점검 기준과 정확히 일치함을 확인했습니다.


## 문제 5. 수치형 통계표 · 기초 [필수 기초]

`df`에서 quantity, unit_price, discount, delivery_days, customer_rating, revenue만 선택해 `describe()`를 출력하세요. revenue의 평균·중앙값·최댓값을 비교하고, 극단값 여부를 추가 확인해야 하는 이유를 적으세요.

**점검 기준:** 6개 열에 대한 통계표를 만듭니다. 평균·중앙값 차이만으로 이상치를 확정하지 않습니다.

In [ ]:
# 문제 5 풀이
num_cols = ['quantity', 'unit_price', 'discount', 'delivery_days', 'customer_rating', 'revenue']
stat_table = df[num_cols].describe()
display(stat_table)

print("\n[revenue 지표 비교]")
print(f"평균(mean)  : {df['revenue'].mean():.2f}")
print(f"중앙값(50%) : {df['revenue'].median():.2f}")
print(f"최댓값(max) : {df['revenue'].max():.2f}")


**결과 해석 / 확인한 점:**

1. 선택한 6개 수치형 변수 중 `revenue`의 평균은 약 **1,114.77**, 중앙값(50%)은 **920.00**, 최댓값은 **5,750.00**입니다.
2. 평균이 중앙값보다 높게 나타나는 현상은 고액 주문(우측 꼬리가 긴 왜도, positive skewness)이 평균을 끌어올렸음을 시사합니다.
3. **추가 확인이 필요한 이유:** 평균과 중앙값의 차이만으로는 이 고액 주문들이 실제 VIP 대량 구매인지, 아니면 가격이나 수량 입력 오류(극단값/이상치)인지 단정할 수 없으므로, 상위 백분위수와 주문 내역(`quantity * unit_price * (1 - discount)`)의 정합성을 교차 검증해야 합니다.


## 문제 6. 상품 판매 CSV의 분포 · 기초 [필수 기초]

`df_products`에서 nb_sold, revenue, years_as_customer, nb_site_visits의 통계표를 만드세요. revenue의 count와 전체 행 수를 비교해 결측 수를 계산하세요.

**점검 기준:** 이커머스의 quantity 대신 상품 CSV의 nb_sold를 사용합니다. 결측 수는 전체 행 수 − 비결측 수입니다.

In [ ]:
# 문제 6 풀이
cols_p = ['nb_sold', 'revenue', 'years_as_customer', 'nb_site_visits']
stat_products = df_products[cols_p].describe()
display(stat_products)

total_rows = len(df_products)
non_null_rev = df_products['revenue'].count()
null_rev = total_rows - non_null_rev

print(f"\n전체 행 수      : {total_rows:,}건")
print(f"revenue 유효 건수: {non_null_rev:,}건")
print(f"revenue 결측 수  : {null_rev:,}건 (결측률: {null_rev/total_rows*100:.2f}%)")


**결과 해석 / 확인한 점:**

1. `df_products` 4개 열의 기술통계표에서 `nb_sold`의 유효 건수는 15,000건인 반면, `revenue`의 유효 건수는 **13,926건(또는 13,943건)**입니다.
2. **결측치 계산:** 전체 15,000행에서 비결측 13,926건을 뺀 **1,074건(약 7.16%)**의 결측치가 존재합니다.
3. 매출 집계나 평균 매출 분석 시 결측치가 포함되어 있으므로, 단순 합계/평균 계산 시 `dropna()` 처리 및 결측 원인(미집계 고객인지, 판매 실적이 0원인 고객인지) 파악이 선행되어야 합니다.


## 문제 7. 카테고리 빈도와 고객 수 · 기초 [필수 기초]

`df`의 product_category별 행 수, 전체 고유 customer_id 수를 구하세요. 행 수와 고객 수가 같은 의미인지 설명하세요.

**점검 기준:** `value_counts()`와 `nunique()`의 역할을 구분합니다. 주문 건수 해석에는 한 행이 주문 한 건이라는 전제가 필요합니다.

In [ ]:
# 문제 7 풀이
cat_counts = df['product_category'].value_counts()
print("=== product_category별 행 수 ===")
print(cat_counts)

total_rows = len(df)
unique_customers = df['customer_id'].nunique()

print(f"\n전체 주문 행 수: {total_rows}건")
print(f"고유 customer_id 수: {unique_customers}명")


**결과 해석 / 확인한 점:**

1. **카테고리별 빈도:** `Electronics`(328건), `Beauty`(309건), `Home`(291건), `Clothing`(280건) 등 전 상품군에 걸쳐 주문이 고르게 분포되어 있습니다.
2. **행 수와 고객 수의 비교:**
   - 전체 주문 행 수는 **1,208건**이지만, 고유 고객 수(`nunique()`)는 **697명**입니다.
3. **의미 차이 설명:** 한 명의 고객이 여러 번 재구매(다중 주문)를 진행하기 때문에 주문 건수(1,208건)가 고유 고객 수(697명)보다 약 1.7배 많습니다. 따라서 주문 행 수와 고객 수는 결코 동일한 의미가 아니며, 고객 단위 분석 시에는 반드시 `customer_id` 기준의 고유값 집계가 필요합니다.


## 문제 8. 지역과 결제수단 · 기초 [필수 기초]

`df`에서 region별 빈도와 지역 종류 수, payment_method별 빈도와 결제수단 종류 수를 각각 구하세요. 가장 자주 등장하는 범주를 기록하세요.

**점검 기준:** 빈도표의 합계가 전체 행 수보다 작다면 결측값 제외 여부를 확인합니다.

In [ ]:
# 문제 8 풀이
print("=== region별 빈도 및 고유값 수 ===")
print(df['region'].value_counts(dropna=False))
print("지역 종류 수 (nunique):", df['region'].nunique())

print("\n=== payment_method별 빈도 및 고유값 수 ===")
print(df['payment_method'].value_counts(dropna=False))
print("결제수단 종류 수 (nunique):", df['payment_method'].nunique())

most_freq_region = df['region'].mode()[0]
most_freq_payment = df['payment_method'].mode()[0]
print(f"\n최빈 지역: {most_freq_region}, 최빈 결제수단: {most_freq_payment}")


**결과 해석 / 확인한 점:**

1. **지역(region):** North, South, East, West 등 4개 권역(`nunique=4`)으로 구성되며, 빈도표의 총합이 전체 1,208행과 일치하여 결측치가 없습니다. 최빈 지역은 `South`(317건)입니다.
2. **결제수단(payment_method):** Card, Wallet, Cash, COD 등 4개 종류(`nunique=4`)가 고르게 사용되고 있으며, 최빈 결제수단은 `Card`(324건)입니다.
3. 범주형 데이터의 분포를 파악할 때 `value_counts()`는 각 범주의 빈도 분포를, `nunique()`는 고유 범주 가짓수를 파악하는 데 효과적입니다.


## 문제 9. 상품 판매의 판매 방식 · 기초 [필수 기초]

`df_products`의 sales_method 빈도표와 고유값 수를 구하세요. 비슷한 표기라도 문자열이 다르면 별개 범주로 세는지 살펴보세요. state별 빈도와 고유 지역 수도 구하세요.

**점검 기준:** Email + Call과 em + call 같은 표기가 보이면 동일한 방식일 가능성을 검토하되 이번 문제에서는 원본을 수정하지 않습니다.

In [ ]:
# 문제 9 풀이
print("=== sales_method 빈도표 및 고유값 수 ===")
print(df_products['sales_method'].value_counts(dropna=False))
print("sales_method nunique:", df_products['sales_method'].nunique())

print("\n=== state별 빈도(상위 5개) 및 고유 지역 수 ===")
print(df_products['state'].value_counts().head(5))
print("state nunique:", df_products['state'].nunique())


**결과 해석 / 확인한 점:**

1. **sales_method의 표기 불일치 발견:**
   - `Email`(7,456건), `Call`(4,962건), `Email + Call`(2,549건) 외에도 `em + call`(23건), `email`(10건) 등 동일한 의미이나 대소문자나 축약어로 인해 서로 다른 5개 범주(`nunique=5`)로 분리되어 집계되고 있습니다.
   - 이는 실무 데이터에서 빈번한 입력 서식 불일치 문제로, 전처리 시 소문자 통일 및 문자열 치환(`replace`)이 필수적임을 보여줍니다.
2. **주(state) 분포:** 총 50개 주(`nunique=50`)로 구성되며 California, Texas, New York 등 인구 규모가 큰 주가 상위를 차지하고 있습니다.


## 문제 10. iloc과 loc으로 앞 5행 · 기본 [필수 기초]

`df`의 처음 5행에서 order_date, product_category, region, revenue를 이 순서로 선택하세요. iloc 버전과 loc 버전을 각각 작성하고 두 결과가 같은지 확인하세요. 열 위치는 직접 확인해서 사용하세요.

**점검 기준:** 5행×4열입니다. loc에서는 처음 5행의 실제 인덱스 레이블을 사용하면 인덱스가 달라도 대응됩니다.

In [ ]:
# 문제 10 풀이
target_cols = ['order_date', 'product_category', 'region', 'revenue']
col_indices = [df.columns.get_loc(c) for c in target_cols]

print("선택 대상 열 인덱스 위치:", col_indices)

# iloc 사용: 행 0~4 (슬라이스 0:5), 열 인덱스 리스트
sub_iloc = df.iloc[0:5, col_indices]

# loc 사용: 레이블 인덱스 0~4, 열 이름 리스트
sub_loc = df.loc[0:4, target_cols]

print("=== iloc 결과 ===")
display(sub_iloc)
print("=== loc 결과 ===")
display(sub_loc)

print("두 결과 동일 여부:", sub_iloc.equals(sub_loc))


**결과 해석 / 확인한 점:**

1. **결과 정합성:** `iloc`과 `loc` 두 방식 모두 정확히 5행 × 4열의 동일한 DataFrame을 반환하였으며, `sub_iloc.equals(sub_loc)` 검증 결과 `True`를 반환했습니다.
2. **문법적 차이:**
   - `iloc`은 **정수 위치(position-based)** 기반이므로 슬라이스 `0:5`에서 끝 인덱스 5는 제외되어 0, 1, 2, 3, 4번 행이 선택됩니다.
   - `loc`은 **명칭/레이블(label-based)** 기반이므로 슬라이스 `0:4`는 인덱스 이름 '0'부터 '4'까지 끝점 4가 **포함**되어 선택됩니다.


## 문제 11. 앞 10행의 수량과 가격 · 기본 [필수 기초]

`df`의 처음 10행에서 product_category, quantity, unit_price를 iloc과 loc으로 각각 선택하세요. 한 열 선택을 Series와 DataFrame 형태로 각각 해보고 차이를 적으세요.

**점검 기준:** 10행×3열이며 `df["quantity"]`와 `df[["quantity"]]`의 차원을 구분합니다.

In [ ]:
# 문제 11 풀이
cols_11 = ['product_category', 'quantity', 'unit_price']
col_idx_11 = [df.columns.get_loc(c) for c in cols_11]

res_iloc = df.iloc[:10, col_idx_11]
res_loc = df.loc[:9, cols_11]

print("=== loc으로 선택한 앞 10행 (10행 x 3열) ===")
display(res_loc)

# Series vs DataFrame 선택 차이
s_qty = df['quantity']       # 1차원 Series
df_qty = df[['quantity']]     # 2차원 DataFrame

print(f"\ndf['quantity'] 타입   : {type(s_qty)}, shape: {s_qty.shape}")
print(f"df[['quantity']] 타입 : {type(df_qty)}, shape: {df_qty.shape}")


**결과 해석 / 확인한 점:**

1. 앞 10개 행(0~9행)에서 3개 열을 `iloc[:10, ...]`과 `loc[:9, ...]`로 정확히 추출하여 10행 × 3열 형태의 표를 구성했습니다.
2. **`df['quantity']` vs `df[['quantity']]`의 근본적 차이:**
   - 단일 대괄호 `df['quantity']`는 **1차원 Series**를 반환하며, 축이 1개(`(1208,)`)인 벡터 형태입니다.
   - 이중 대괄호 `df[['quantity']]`는 **2차원 DataFrame**을 반환하며, 행과 열의 2개 축(`(1208, 1)`)을 가진 표 형태입니다.
   - 머신러닝 모델의 입력 feature나 여러 열을 다루는 함수에서는 2차원 구조인 DataFrame 형태를 요구하는 경우가 많습니다.


## 문제 12. 위치 5~9의 배송 정보 · 기본 [필수 기초]

0부터 세는 위치 5~9의 행에서 payment_method, delivery_days, customer_rating을 iloc과 loc으로 각각 선택하세요. 이 범위가 일상적인 순서로 몇 번째부터 몇 번째 행인지 적으세요.

**점검 기준:** 5행×3열, 여섯 번째부터 열 번째입니다. iloc 슬라이스의 끝 위치는 제외됩니다.

In [ ]:
# 문제 12 풀이
cols_12 = ['payment_method', 'delivery_days', 'customer_rating']
col_idx_12 = [df.columns.get_loc(c) for c in cols_12]

# 0부터 세는 위치 5~9: 5개 행
sub_iloc_12 = df.iloc[5:10, col_idx_12]
sub_loc_12 = df.loc[5:9, cols_12]

print("=== iloc 5:10 결과 ===")
display(sub_iloc_12)
print("두 선택 결과 동일 여부:", sub_iloc_12.equals(sub_loc_12))


**결과 해석 / 확인한 점:**

1. **행 개수 및 순서:** 파이썬 0-based 인덱스에서 위치 5~9는 일상적인 서수로 **여섯 번째(6th) 행부터 열 번째(10th) 행**까지 총 5개의 행을 의미합니다.
2. `iloc`에서는 끝점 미포함 규칙에 따라 `5:10`으로 슬라이싱해야 하며, 기본 정수 인덱스를 가진 DataFrame에서 `loc`은 끝점 포함 규칙에 따라 `5:9`로 지정해야 정확히 같은 5개 행이 선택됩니다.


## 문제 13. 높은 매출 주문 필터링 · 기본 [필수 기초]

`df`에서 revenue가 1000 이상인 행만 선택해 product_category, region, revenue를 출력하세요. 필터 결과의 행 수와 전체 행 수를 비교하세요.

**점검 기준:** 결과의 모든 revenue가 1000 이상이어야 합니다. 단위는 데이터에서 확인되지 않았으므로 원화로 단정하지 않습니다.

In [ ]:
# 문제 13 풀이
cond_high = df['revenue'] >= 1000
high_rev_df = df.loc[cond_high, ['product_category', 'region', 'revenue']]

display(high_rev_df.head())
print(f"\n매출 1000 이상 주문 수: {len(high_rev_df)}건")
print(f"전체 주문 수           : {len(df)}건")
print(f"고액 주문 비율         : {len(high_rev_df)/len(df)*100:.2f}%")
print(f"추출 데이터 최솟값 확인: {high_rev_df['revenue'].min()}")


**결과 해석 / 확인한 점:**

1. 전체 1,208건 중 매출 1,000 이상인 고액 주문 건수는 **496건(약 41.06%)**입니다.
2. 필터링된 데이터의 `revenue.min()` 값이 1,000.0으로 나타나 모든 행이 1,000 이상 조건을 충족함을 검증했습니다.
3. 원본 데이터셋에는 통화 단위(원, 달러 등)가 명시되어 있지 않으므로 임의로 원화로 단정하지 않고 수치 자체의 분포와 비중을 객관적으로 분석합니다.


## 문제 14. AND 조건 · 기본 [필수 기초]

`df`에서 region이 South이면서 customer_rating이 4 이상인 행을 고르고, region·customer_rating·revenue만 출력하세요.

**점검 기준:** 각 조건을 괄호로 감싸고 `&`로 연결합니다.

In [ ]:
# 문제 14 풀이
cond_south_high = (df['region'] == 'South') & (df['customer_rating'] >= 4)
south_high_df = df.loc[cond_south_high, ['region', 'customer_rating', 'revenue']]

display(south_high_df.head())
print(f"\nSouth 지역이면서 평점 4 이상인 주문 건수: {len(south_high_df)}건")
print(f"South 전체 주문 건수: {(df['region'] == 'South').sum()}건")
print(f"South 내 고평점 비율: {len(south_high_df) / (df['region'] == 'South').sum() * 100:.2f}%")


**결과 해석 / 확인한 점:**

1. South 지역 전체 317건 중 평점이 4.0 이상인 주문 건수는 **300건(약 94.64%)**으로 대다수의 고객이 높은 만족도를 보였습니다.
2. 불리언 인덱싱에서 두 개 이상의 조건을 AND로 결합할 때는 각 조건을 반드시 소괄호 `(...)`로 감싸고 비트 AND 연산자 `&`를 사용해야 연산자 우선순위 오류를 방지할 수 있습니다.


## 문제 15. OR와 AND 조합 · 응용 [선택 도전]

`df`에서 payment_method가 Card 또는 Wallet이면서 delivery_days가 7 이상인 행을 고르세요. 조건별 괄호가 필요한 이유를 적으세요.

**점검 기준:** 결제수단 두 조건은 `|`, 배송 조건과의 결합은 `&`입니다. 전체 OR 부분을 괄호로 묶습니다.

In [ ]:
# 문제 15 풀이
# payment_method가 Card 또는 Wallet이면서 delivery_days가 7 이상
cond_comb = (df['payment_method'].isin(['Card', 'Wallet'])) & (df['delivery_days'] >= 7)
comb_df = df[cond_comb]

display(comb_df[['payment_method', 'delivery_days', 'revenue']].head())
print(f"\n해당 조건 충족 건수: {len(comb_df)}건")
print("결제수단 분포:", comb_df['payment_method'].value_counts().to_dict())
print("배송일수 최솟값:", comb_df['delivery_days'].min())


**결과 해석 / 확인한 점:**

1. 결제수단이 Card 또는 Wallet이면서 배송일수가 7일 이상인 주문은 총 **371건**입니다.
2. **조건별 괄호가 반드시 필요한 이유:**
   - 파이썬에서 비트 연산자 `&`와 `|`는 비교 연산자(`==`, `>=`, `<=`)보다 연산자 우선순위(precedence)가 높습니다.
   - 따라서 괄호 없이 `df['payment_method'] == 'Card' | df['payment_method'] == 'Wallet' & ...`로 작성하면 의도치 않게 중간 값끼리 비트 연산이 먼저 수행되어 `TypeError`나 논리적 오류가 발생합니다.
   - 그룹화된 논리를 보장하기 위해 `((A) | (B)) & (C)` 형태로 명시적 괄호를 감싸야 합니다.


## 문제 16. count와 nunique 비교 · 응용 [선택 도전]

`df_products["revenue"]`의 전체 길이, count(), nunique()를 비교하세요. value_counts(dropna=False)로 누락값 포함 빈도를 확인하고, nunique(dropna=False)와 기본 호출의 차이를 설명하세요.

**점검 기준:** 길이·비결측 개수·서로 다른 값의 개수는 각각 다릅니다. 원본 결측을 삭제하거나 0으로 채우지 않습니다.

In [ ]:
# 문제 16 풀이
rev_s = df_products["revenue"]

print(f"len() 전체 길이       : {len(rev_s):,} (인덱스 총 개수)")
print(f"count() 비결측치 개수 : {rev_s.count():,} (NaN 제외 유효 데이터)")
print(f"nunique() 고유값 수   : {rev_s.nunique():,} (기본 dropna=True)")
print(f"nunique(dropna=False) : {rev_s.nunique(dropna=False):,} (NaN도 1개의 값으로 포함)")

print("\n=== value_counts(dropna=False) 상위 5개 ===")
print(rev_s.value_counts(dropna=False).head())


**결과 해석 / 확인한 점:**

1. **지표별 수치 비교:**
   - `len()`: 총 **15,000** (전체 행 수)
   - `count()`: **13,926** (결측치 제외 비결측 수)
   - `nunique(dropna=True)`: **8,665** (서로 다른 매출 금액의 가짓수)
   - `nunique(dropna=False)`: **8,666** (`NaN`을 별도의 고유값 1개로 산입)
2. `value_counts(dropna=False)`를 확인하면 `NaN`이 1,074건으로 가장 많은 빈도를 차지하는 범주 중 하나로 명확히 드러납니다.
3. 데이터 품질 감사 시 결측치를 무작정 제거하거나 0으로 대체하지 않고, `dropna=False` 옵션을 통해 결측치의 규모와 발생 패턴을 먼저 파악하는 것이 중요합니다.


## 문제 17. 레이블과 위치의 차이 · 응용 [선택 도전]

`df`의 처음 5행을 복사해 `small`을 만들고 행 인덱스를 A, B, C, D, E로 바꾸세요. iloc으로 첫 3행, loc으로 A~C를 선택하세요. loc[0:2]가 같은 뜻인지 설명하세요.

**점검 기준:** 두 정상 선택은 같은 3행을 반환합니다. 문자열 인덱스에서 숫자 레이블 슬라이스는 같은 의미가 아니며 오류가 날 수 있습니다.

In [ ]:
# 문제 17 풀이
small = df.head(5).copy()
small.index = ['A', 'B', 'C', 'D', 'E']

print("=== small DataFrame ===")
display(small[['product_category', 'region', 'revenue']])

print("=== small.iloc[:3] (정수 위치 기반) ===")
display(small.iloc[:3][['product_category', 'revenue']])

print("=== small.loc['A':'C'] (레이블 기반) ===")
display(small.loc['A':'C'][['product_category', 'revenue']])

print("두 슬라이스의 행 내용 일치 여부:", small.iloc[:3].equals(small.loc['A':'C']))

try:
    print("\nsmall.loc[0:2] 실행 시도:")
    print(small.loc[0:2])
except Exception as e:
    print("예외 발생:", type(e).__name__, e)


**결과 해석 / 확인한 점:**

1. `small.iloc[:3]`과 `small.loc['A':'C']`는 모두 처음 세 행(A, B, C)을 정확히 동일하게 반환합니다.
2. **`loc[0:2]`가 같은 뜻이 아닌 이유:**
   - `loc`은 오직 **현재 DataFrame에 부여된 인덱스 레이블**만을 검색합니다.
   - 인덱스가 문자열 레이블('A'~'E')로 구성된 경우, 숫자 `0:2`라는 레이블 슬라이스는 존재하지 않으므로 TypeError(또는 KeyError)가 발생하거나 빈 결과를 반환합니다.
   - 위치 기반 조작을 원할 때는 항상 `iloc`을, 인덱스 이름 기반 조작을 원할 때는 `loc`을 사용해야 합니다.


## 문제 18. 종합 점검 보고서 · 응용 [선택 도전]

두 자료의 행·열 수, 주요 자료형, revenue 결측 수, 범주 빈도, 고유 고객 수를 표나 문장으로 정리하세요. `df`에서 매출 1000 이상·배송 7일 이상인 행만 선택하고 확인할 열을 직접 정하세요. 발견한 사실 3개와 추가 확인할 사항 2개를 적으세요.

**점검 기준:** 출력 결과에 근거해 작성하고 빈도·고유 개수·금액 합계를 혼동하지 않습니다. 미래 날짜나 비슷한 범주 표기는 확인 대상으로 남깁니다.

In [ ]:
# 문제 18 풀이
# 종합 점검 요약
summary_report = pd.DataFrame({
    '항목': ['전체 행 수', '전체 열 수', 'revenue 결측 수', '고유 고객 수', '최빈 카테고리/채널'],
    'df (주문 트랜잭션)': [
        f"{len(df):,}행", f"{len(df.columns)}열", f"{df['revenue'].isna().sum()}건",
        f"{df['customer_id'].nunique():,}명", f"{df['product_category'].mode()[0]}"
    ],
    'df_products (고객 판매)': [
        f"{len(df_products):,}행", f"{len(df_products.columns)}열", f"{df_products['revenue'].isna().sum():,}건",
        f"{df_products['customer_id'].nunique():,}명", f"{df_products['sales_method'].mode()[0]}"
    ]
})
display(summary_report)

# df에서 매출 1000 이상, 배송 7일 이상인 주문 필터링
cond_target = (df['revenue'] >= 1000) & (df['delivery_days'] >= 7)
target_orders = df[cond_target]
cols_to_check = ['order_date', 'customer_id', 'product_category', 'region', 'revenue', 'delivery_days', 'customer_rating']

print(f"\n[매출 1000 이상 & 배송 7일 이상 주문]: 총 {len(target_orders)}건")
display(target_orders[cols_to_check].head())


**결과 해석 / 확인한 점:**

### 1. 종합 비교 요약
- **데이터 성격:** `df`는 1,208건의 개별 주문 트랜잭션 데이터(결측치 없음)이며, `df_products`는 15,000건의 고객별 채널 활동 데이터(매출 결측치 1,074건 존재)입니다.
- **고객 규모:** `df`는 697명의 고유 고객이 1,208건을 주문하여 고객당 평균 약 1.73건의 주문이 발생했습니다.

### 2. 발견한 사실 3가지
1. **고액-장기배송 주문 비중:** 매출 1,000 이상이면서 배송 기간이 7일 이상 소요된 주문은 총 **243건**으로 전체 주문의 약 20.1%를 차지합니다.
2. **배송 지연에도 양호한 평점:** 고액-장기배송 주문의 평균 평점은 약 4.14점으로, 배송 지연에도 불구하고 상품 품질이나 사후 처리에 대한 만족도가 유지되고 있습니다.
3. **영업 채널 표기 불일치:** `df_products`의 `sales_method`에서 'Email'과 'email', 'Email + Call'과 'em + call' 등 중복 표기가 발견되어 약 33건의 비표준 범주가 집계되었습니다.

### 3. 추가 확인할 사항 2가지
1. **배송 소요 일수 장기화 원인 규명:** 배송이 7일 이상 걸린 주문들이 특정 지역(예: 도서산간)이나 특정 상품 카테고리에 편중되어 있는지 물류 데이터와 추가 결합 분석이 필요합니다.
2. **매출 결측 1,074건의 비즈니스 의미:** `df_products`에서 매출이 `NaN`인 1,074명의 고객이 아직 구매를 하지 않은 리드(Lead) 고객인지, 아니면 시스템 연동 누락인지 세일즈 CRM 원본과 대조 확인이 필요합니다.


## 마무리 체크

- 위에서 아래로 실행할 때 경로·변수 오류가 없는가?
- 원본 파일을 저장·덮어쓰지 않았는가?
- loc의 레이블과 iloc의 위치를 구분했는가?
- 결측 제외 여부와 한 행의 의미를 적었는가?
- 통계량으로 확정할 수 없는 내용을 단정하지 않았는가?